# Final analysis

Generates the figures used in the thesis: bootstrap CIs, FP/FN examples, feature importance from the ODE params, and CODE-15 vs SaMi-Trop subgroup metrics. Loads `approach10_mcsharry_best.pt` and `preprocessed_cache_brazil.h5`.

In [5]:
import math
import os
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

sns.set_context("paper", font_scale=1.2)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.family"] = "serif"

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

Device: mps


### 1. Definition of the McSharry PINN Model

In [6]:
class AdapResBlock1D(nn.Module):
    def __init__(self, ch, kernel_size=15, dropout=0.2):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Sequential(
            nn.Conv1d(ch, ch, kernel_size, padding=pad, bias=False),
            nn.GroupNorm(8, ch),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(ch, ch, kernel_size, padding=pad, bias=False),
            nn.GroupNorm(8, ch),
        )
        self.alpha = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return x + self.alpha * self.conv(x)

class McSharryPINN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        c = cfg["base_ch"]
        self.stem = nn.Sequential(
            nn.Conv1d(cfg["in_channels"], c, kernel_size=cfg["kernel_size"], padding=cfg["kernel_size"] // 2, bias=False),
            nn.GroupNorm(8, c),
            nn.GELU(),
        )
        self.blocks = nn.ModuleList([AdapResBlock1D(c, cfg["kernel_size"], cfg["dropout"]) for _ in range(cfg["num_adap_blocks"])])
        self.down = nn.Conv1d(c, c, kernel_size=4, stride=2, padding=1, bias=False)
        self.proj = nn.Sequential(nn.GroupNorm(8, c), nn.GELU())
        
        self.ode_head = nn.Linear(c, 17)
        self.clf = nn.Sequential(
            nn.Linear(17, 32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x, return_params=False):
        h = self.stem(x)
        for blk in self.blocks:
            h = blk(h)
        h = self.down(h)
        h = self.proj(h)
        ctx = h.mean(dim=-1)
        
        raw = self.ode_head(ctx)
        
        f = torch.sigmoid(raw[:, 0:1]) * 2.0 + 0.5
        z0 = torch.tanh(raw[:, 1:2]) * 0.5
        
        A = torch.tanh(raw[:, 2:7]) * 5.0
        b = F.softplus(raw[:, 7:12]) * 1.0 + 0.01
        
        theta_R = torch.zeros_like(raw[:, 14:15])
        
        delta_QR = torch.sigmoid(raw[:, 13:14]) * (math.pi / 2)
        delta_PQ = torch.sigmoid(raw[:, 12:13]) * (math.pi / 2)
        delta_RS = torch.sigmoid(raw[:, 15:16]) * (math.pi / 2)
        delta_ST = torch.sigmoid(raw[:, 16:17]) * (math.pi / 2)
        
        theta_Q = theta_R - delta_QR
        theta_P = theta_Q - delta_PQ
        theta_S = theta_R + delta_RS
        theta_T = theta_S + delta_ST
        
        theta_all = torch.cat([theta_P, theta_Q, theta_R, theta_S, theta_T], dim=-1)
        theta = torch.atan2(torch.sin(theta_all), torch.cos(theta_all))
        
        params = torch.cat([f, z0, A, b, theta], dim=-1)
        
        clf_in = params
        logits = self.clf(clf_in).squeeze(-1)
        
        if return_params:
            return logits, params
        return logits

### 2. Loading Data and Running Predictions on the Test Set

In [7]:
CFG = {
    "preprocessed_cache": "preprocessed_cache_brazil.h5",
    "test_fraction": 0.15,
    "val_fraction": 0.15,
    "random_seed": 42,
    "in_channels": 12,
    "base_ch": 64,
    "num_adap_blocks": 4,
    "kernel_size": 15,
    "dropout": 0.0,
}

cache_file = CFG["preprocessed_cache"]

from sklearn.model_selection import GroupShuffleSplit

class CachedChagasDataset(Dataset):
    def __init__(self, cache_path, indices):
        self.cache_path = cache_path
        self.indices = np.sort(indices)
        self._file = None
        with h5py.File(cache_path, "r") as f:
            self.labels = f["labels"][self.indices]

    def __len__(self):
        return len(self.indices)

    def _f(self):
        if self._file is None:
            self._file = h5py.File(self.cache_path, "r")
        return self._file

    def __getitem__(self, idx):
        f = self._f()
        ri = self.indices[idx]
        x = torch.from_numpy(f["signals"][ri]).float()
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y, ri

if os.path.exists(cache_file):
    with h5py.File(cache_file, "r") as f:
        n_total = f["labels"].shape[0]
        all_labels = f["labels"][:]
        all_exam_ids = f["exam_ids"][:] if "exam_ids" in f else np.arange(n_total)
    print(f"Samples: {n_total:,} | Chagas+ {100*all_labels.mean():.1f}%")

    code15_dir = "4916206"
    samitrop_dir = "sami-trop"

    df_code15 = pd.read_csv(os.path.join(code15_dir, "exams.csv")) if os.path.exists(os.path.join(code15_dir, "exams.csv")) else pd.DataFrame()
    df_samitrop = pd.read_csv(os.path.join(samitrop_dir, "exams.csv")) if os.path.exists(os.path.join(samitrop_dir, "exams.csv")) else pd.DataFrame()

    if not df_code15.empty:
        df_code15["source"] = "code15"
    if not df_samitrop.empty:
        df_samitrop["source"] = "samitrop"

    df_meta = pd.concat([df_code15, df_samitrop], ignore_index=True)

    if not df_meta.empty and "exam_id" in df_meta.columns and "patient_id" in df_meta.columns:
        exam_to_patient = dict(zip(df_meta["exam_id"], df_meta["patient_id"]))
        exam_to_source = dict(zip(df_meta["exam_id"], df_meta["source"]))

        patient_ids = np.array([exam_to_patient.get(eid, eid) for eid in all_exam_ids])
        sources = np.array([exam_to_source.get(eid, "unknown") for eid in all_exam_ids])

        test_mask = (sources == "samitrop")
        test_indices = np.where(test_mask)[0]

        train_val_indices = np.where(~test_mask)[0]

        gss = GroupShuffleSplit(n_splits=1, test_size=CFG["val_fraction"] / (1.0 - CFG["test_fraction"]), random_state=CFG["random_seed"])
        train_idx_rel, val_idx_rel = next(gss.split(train_val_indices, groups=patient_ids[train_val_indices]))

        train_indices = train_val_indices[train_idx_rel]
        val_indices = train_val_indices[val_idx_rel]
    else:
        print("Warning: Could not load patient metadata. Falling back to random split.")
        n_test = int(n_total * CFG["test_fraction"])
        n_val = int(n_total * CFG["val_fraction"])
        n_train = n_total - n_val - n_test
        rng = np.random.RandomState(CFG["random_seed"])
        perm = rng.permutation(n_total)
        train_indices = perm[:n_train]
        val_indices = perm[n_train : n_train + n_val]
        test_indices = perm[n_train + n_val :]

    print(f"Split sizes -> Train: {len(train_indices)}, Val: {len(val_indices)}, Test: {len(test_indices)}")

    test_loader = DataLoader(CachedChagasDataset(cache_file, test_indices), batch_size=32, shuffle=False)
else:
    print("Warning: data file not found. Calculations will be skipped.")
    test_loader = []

model = McSharryPINN(CFG).to(device)
weights_path = "approach10_mcsharry_best.pt"
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print(f"Loaded weights from {weights_path}")
else:
    print("Warning: weights file not found. Inference on the test set may be random.")

model.eval()
y_true, y_prob, extracted_params, original_indices, signals_cache = [], [], [], [], []
with torch.no_grad():
    for x, y, ri in tqdm(test_loader, desc="Test Inference"):
        logits, params = model(x.to(device), return_params=True)
        y_true.extend(y.numpy())
        y_prob.extend(torch.sigmoid(logits).cpu().numpy())
        extracted_params.append(params.cpu().numpy())
        original_indices.extend(ri.numpy())
        signals_cache.append(x.numpy())

if len(y_true) > 0:
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    extracted_params = np.concatenate(extracted_params, axis=0)
    original_indices = np.array(original_indices)
    signals_cache = np.concatenate(signals_cache, axis=0)

Samples: 49,152 | Chagas+ 16.7%
Split sizes -> Train: 39109, Val: 8412, Test: 1631
Loaded weights from approach10_mcsharry_best.pt


Test Inference: 100%|██████████| 51/51 [00:05<00:00,  8.86it/s]


### 3. Medical Statistics: Bootstrapping for Confidence Intervals (95% CI)

In [ ]:
def calculate_tpr_at_capacity(y_t, y_p, capacity_ratio=0.05):
    n_positives = np.sum(y_t)
    if n_positives == 0: return 0.0
    M = int(np.floor(len(y_t) * capacity_ratio))
    if M == 0: return 0.0
    sorted_indices = np.argsort(y_p)[::-1]
    top_m_indices = sorted_indices[:M]
    return float(np.sum(y_t[top_m_indices]) / n_positives)

def compute_ci_bootstrap(y_true, y_prob, n_bootstraps=1000, alpha=0.95):
    if len(y_true) == 0:
        return {}
        
    rng = np.random.RandomState(42)
    auroc_scores = []
    auprc_scores = []
    cs_scores = []
    
    for i in tqdm(range(n_bootstraps), desc="Bootstrapping"):
        indices = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[indices])) < 2:
            continue
            
        auroc_scores.append(roc_auc_score(y_true[indices], y_prob[indices]))
        auprc_scores.append(average_precision_score(y_true[indices], y_prob[indices]))
        cs_scores.append(calculate_tpr_at_capacity(y_true[indices], y_prob[indices]))
        
    results = {}
    for name, scores in zip(["AUROC", "AUPRC", "Challenge Score"], [auroc_scores, auprc_scores, cs_scores]):
        scores = np.array(scores)
        if len(scores) == 0:
            print(f"Warning: no valid bootstrap samples for {name} (all iterations had only one class).")
            results[name] = {"mean": float("nan"), "lower": float("nan"), "upper": float("nan")}
            continue
        p_lower = ((1.0 - alpha) / 2.0) * 100
        p_upper = (alpha + ((1.0 - alpha) / 2.0)) * 100
        results[name] = {
            "mean": np.mean(scores),
            "lower": np.percentile(scores, p_lower),
            "upper": np.percentile(scores, p_upper)
        }
    return results

if 'y_true' in locals() and len(y_true) > 0:
    ci_results = compute_ci_bootstrap(y_true, y_prob, n_bootstraps=1000)
    print("\n=== Bootstrapping Results (95% CI) ===")
    for metric, vals in ci_results.items():
        print(f"{metric}: {vals['mean']:.4f} [95% CI: {vals['lower']:.4f} - {vals['upper']:.4f}]")

### 4. Feature importance from the PINN parameters

Logistic regression on the extracted McSharry parameters, to see which ones the classifier leans on.

In [ ]:
if 'extracted_params' in locals() and len(extracted_params) > 0:
    feature_names = [
        "Heart Rate (f)", "Baseline (z0)", 
        "Amp P", "Amp Q", "Amp R", "Amp S", "Amp T",
        "Width P", "Width Q", "Width R", "Width S", "Width T",
        "Phase P", "Phase Q", "Phase R", "Phase S", "Phase T"
    ]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(extracted_params)

    lr_model = LogisticRegression(penalty='l2', C=1.0, max_iter=1000, random_state=42)
    lr_model.fit(X_scaled, y_true)

    importances = lr_model.coef_[0]
    indices = np.argsort(np.abs(importances))[::-1]

    plt.figure(figsize=(10, 6))
    sns.barplot(x=importances[indices], y=np.array(feature_names)[indices], palette="coolwarm")
    plt.title("McSharry PINN - Feature Importance (Logistic Regression Coefficients)", fontsize=14)
    plt.xlabel("Coefficient Magnitude (Impact on Chagas+ Prediction)")
    plt.tight_layout()
    plt.savefig("pinn_feature_importance.png", dpi=300)
    plt.show()

### 5. Failure analysis

Top 3 false positives and top 3 false negatives, ranked by predicted probability.

In [ ]:
if 'y_true' in locals() and len(y_true) > 0:
    if 'original_indices' in locals() and len(original_indices) > 0:
        with h5py.File(cache_file, 'r') as f:
            test_exam_ids = f['exam_ids'][:][original_indices]
    else:
        test_exam_ids = None

    fp_indices = np.where(y_true == 0)[0]
    fp_sorted = fp_indices[np.argsort(y_prob[fp_indices])[::-1]]
    
    fn_indices = np.where(y_true == 1)[0]
    fn_sorted = fn_indices[np.argsort(y_prob[fn_indices])]
    
    def plot_ecg_example(signal, title):
        plt.figure(figsize=(12, 4))
        time_axis = np.linspace(0, 10, signal.shape[1])
        for lead in range(min(12, signal.shape[0])):
            plt.plot(time_axis, signal[lead] + lead*3, alpha=0.8, linewidth=0.5)
        plt.title(title)
        plt.xlabel("Time (s)")
        plt.yticks(np.arange(0, 12*3, 3), [f"L{i+1}" for i in range(12)])
        plt.tight_layout()
        plt.show()

    print("--- False Positives Analysis (Top 3 worst) ---")
    for idx in fp_sorted[:3]:
        exam_id_str = f" | Exam ID: {test_exam_ids[idx]}" if test_exam_ids is not None else ""
        title = f"False Positive: Prediction={y_prob[idx]:.3f} (True=Chagas Negative){exam_id_str}"
        plot_ecg_example(signals_cache[idx], title)
        
    print("--- False Negatives Analysis (Top 3 worst) ---")
    for idx in fn_sorted[:3]:
        exam_id_str = f" | Exam ID: {test_exam_ids[idx]}" if test_exam_ids is not None else ""
        title = f"False Negative: Prediction={y_prob[idx]:.3f} (True=Chagas Positive){exam_id_str}"
        plot_ecg_example(signals_cache[idx], title)

### 6. Domain shift: CODE-15 vs SaMi-Trop

Split the test set by source CSV and report metrics per subgroup. SaMi-Trop is 100% Chagas+, so only sensitivity is meaningful there.

In [ ]:
if 'original_indices' in locals() and len(original_indices) > 0:
    with h5py.File(cache_file, 'r') as f:
        test_exam_ids = f['exam_ids'][:][original_indices]

    try:
        sami_exams = pd.read_csv('sami-trop/exams.csv')['exam_id'].values
        code_exams = pd.read_csv('4916206/exams.csv')['exam_id'].values
        
        sami_set = set(sami_exams)
        code_set = set(code_exams)
        
        source_mask_sami = np.array([eid in sami_set for eid in test_exam_ids])
        source_mask_code = np.array([eid in code_set for eid in test_exam_ids])
        
        print("--- Domain Shift Analysis ---")
        print(f"Total Test Samples: {len(test_exam_ids)}")
        print(f"  - CODE-15% Samples: {np.sum(source_mask_code)}")
        print(f"  - SaMi-Trop Samples: {np.sum(source_mask_sami)}")
        print(f"  - Unmapped: {len(test_exam_ids) - np.sum(source_mask_code) - np.sum(source_mask_sami)}")
        
        print("\n1. Performance on CODE-15%:")
        if np.sum(source_mask_code) > 0:
            y_t_code = y_true[source_mask_code]
            y_p_code = y_prob[source_mask_code]
            if len(np.unique(y_t_code)) >= 2:
                print(f"  AUROC: {roc_auc_score(y_t_code, y_p_code):.4f}")
                print(f"  AUPRC: {average_precision_score(y_t_code, y_p_code):.4f}")
                print(f"  Challenge Score: {calculate_tpr_at_capacity(y_t_code, y_p_code):.4f}")
            else:
                print("  Not enough classes to compute metrics.")
        else:
            print("  Not enough data.")
            
        print("\n2. Performance on SaMi-Trop (Sensitivity Evaluation):")
        if np.sum(source_mask_sami) > 0:
            y_p_sami = (y_prob[source_mask_sami] > 0.5).astype(int)
            sens_sami = np.mean(y_p_sami == y_true[source_mask_sami])
            print(f"  Sensitivity: {sens_sami:.4f} (at default threshold 0.5)")
            print(f"  Positives detected: {np.sum(y_p_sami == 1)} out of {len(y_p_sami)}")
        else:
            print("  Not enough data.")
            
    except Exception as e:
        print(f"Error loading metadata for domain shift: {e}")
else:
    print("No test data available for domain shift analysis.")